In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import warnings
# numpy 2.0 + macOS Accelerate prints a harmless "encountered in matmul" warning inside sklearn
warnings.filterwarnings("ignore", message=".*encountered in matmul")

In [2]:
df = pd.read_csv("../Data/placement.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   student_id           550 non-null    object 
 1   cgpa                 550 non-null    float64
 2   twelfth_percent      550 non-null    float64
 3   branch               550 non-null    object 
 4   internships          519 non-null    float64
 5   projects             550 non-null    int64  
 6   backlogs             550 non-null    int64  
 7   prep_hours           454 non-null    float64
 8   communication_score  538 non-null    float64
 9   hostel               550 non-null    object 
 10  placed               550 non-null    object 
 11  package_lpa          517 non-null    float64
dtypes: float64(6), int64(2), object(4)
memory usage: 51.7+ KB


In [3]:
df.head()

,student_id,cgpa,twelfth_percent,branch,internships,projects,backlogs,prep_hours,communication_score,hostel,placed,package_lpa
0,VS2821,7.41,78.5,IT,0.0,6,1,188.0,4.3,No,No,0.00
1,VS3068,7.52,70.0,CSE,2.0,3,0,167.0,9.8,Yes,Yes,10.06
2,VS2940,7.35,78.3,CSE,2.0,0,3,199.0,4.8,Yes,Yes,NaN
3,VS3138,7.86,60.2,CSE,1.0,3,0,135.0,8.0,No,Yes,8.63
4,VS2670,5.63,88.3,CSE,2.0,2,0,132.0,6.3,No,Yes,5.22


In [4]:
df.describe()

,cgpa,twelfth_percent,internships,projects,backlogs,prep_hours,communication_score,package_lpa
count,550.000000,550.000000,519.000000,550.00000,550.000000,454.000000,538.000000,517.000000
mean,7.358255,77.597455,1.302505,2.56000,0.683636,153.151982,6.269331,4.711857
std,0.983022,10.121533,1.181834,1.49164,1.097233,86.055100,1.689150,3.475017
min,5.000000,55.000000,0.000000,0.00000,0.000000,16.000000,1.100000,0.000000
25%,6.690000,70.900000,0.000000,1.00000,0.000000,90.000000,5.200000,0.000000
50%,7.400000,77.900000,1.000000,2.00000,0.000000,134.000000,6.300000,6.010000
75%,7.975000,84.500000,2.000000,4.00000,1.000000,192.750000,7.400000,7.280000
max,10.000000,98.000000,4.000000,6.00000,4.000000,500.000000,10.000000,12.560000


In [5]:
df.describe().columns

Index(['cgpa', 'twelfth_percent', 'internships', 'projects', 'backlogs',
       'prep_hours', 'communication_score', 'package_lpa'],
      dtype='object')

In [6]:
x = df.drop(columns=["student_id",'placed','package_lpa'])
y = df['placed']

In [7]:
nums_col = ['cgpa', 'twelfth_percent', 'internships', 'projects', 'backlogs',
       'prep_hours', 'communication_score']
cat_col = ['branch', 'hostel']

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   student_id           550 non-null    object 
 1   cgpa                 550 non-null    float64
 2   twelfth_percent      550 non-null    float64
 3   branch               550 non-null    object 
 4   internships          519 non-null    float64
 5   projects             550 non-null    int64  
 6   backlogs             550 non-null    int64  
 7   prep_hours           454 non-null    float64
 8   communication_score  538 non-null    float64
 9   hostel               550 non-null    object 
 10  placed               550 non-null    object 
 11  package_lpa          517 non-null    float64
dtypes: float64(6), int64(2), object(4)
memory usage: 51.7+ KB


In [9]:
x = df.drop(columns=['student_id','placed','package_lpa'])
y = df['placed']

In [10]:
for col in nums_col:
    df[col] = df[col].fillna(df[col].mean())
    

In [11]:
df.isnull().sum()

student_id              0
cgpa                    0
twelfth_percent         0
branch                  0
internships             0
projects                0
backlogs                0
prep_hours              0
communication_score     0
hostel                  0
placed                  0
package_lpa            33
dtype: int64

In [12]:
print(cat_col)

['branch', 'hostel']


In [13]:
df['branch'].value_counts()

branch
CSE      167
IT       118
ECE      112
Mech     103
Civil     50
Name: count, dtype: int64

In [14]:
df['hostel'].value_counts()

hostel
Yes    317
No     233
Name: count, dtype: int64

In [15]:
from sklearn.preprocessing import OneHotEncoder


In [16]:
encoder = OneHotEncoder(sparse_output=False)

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df[nums_col + cat_col]
y = df['placed']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
# fit the encoder on train only, then encode both parts
enc_train = pd.DataFrame(encoder.fit_transform(X_train[cat_col]), columns=encoder.get_feature_names_out(cat_col), index=X_train.index)
enc_test = pd.DataFrame(encoder.transform(X_test[cat_col]), columns=encoder.get_feature_names_out(cat_col), index=X_test.index)

X_train = pd.concat([X_train[nums_col], enc_train], axis=1)
X_test = pd.concat([X_test[nums_col], enc_test], axis=1)
X_train.head()

,cgpa,twelfth_percent,internships,projects,backlogs,prep_hours,communication_score,branch_CSE,branch_Civil,branch_ECE,branch_IT,branch_Mech,hostel_No,hostel_Yes
482,7.45,66.5,1.0,2,0,100.0,5.4,0.0,0.0,0.0,1.0,0.0,0.0,1.0
158,7.22,79.8,0.0,5,0,246.0,6.6,0.0,0.0,0.0,1.0,0.0,1.0,0.0
15,7.74,80.6,0.0,6,0,68.0,7.9,0.0,0.0,1.0,0.0,0.0,1.0,0.0
334,7.06,68.8,0.0,3,0,107.0,6.7,0.0,0.0,0.0,1.0,0.0,0.0,1.0
39,7.70,84.0,0.0,2,1,107.0,7.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


In [19]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)
print("Accuracy:", accuracy_score(y_test, model.predict(X_test_scaled)))

Accuracy: 0.8090909090909091
